In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.metrics import mean_squared_error


import sys
# Add the directory where the package is installed
sys.path.append('/home/sestupin/.local/lib/python3.10/site-packages/')
import xgboost as xgb
from xgboost import XGBClassifier


In [2]:
#Training Data.
#Skip the header as it is not a number.
#Each row corresponds to a movie and displays its runtime, IMDB rating, IMDB number of votes,...
X = np.genfromtxt('Training_X.csv', delimiter=',', skip_header=1)  
m, n = X.shape
#Each row of y says if 2.5*movie_budget was greater than worldwide_gross. This is the metric we are using to 
#determine if a movie was successful.
y = np.genfromtxt('Training_Y.csv', delimiter=',', skip_header=1)

In [3]:
#Splitting data to have a part of it for testing and cross-validation
#I am saving 30% of the total data for that purpose.
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.3, random_state=1)

#Then split into test and cross-validation that 30%. Half goes to cv half to test.
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.50, random_state=1)

#Normalize the data so that features are on the same range.
#scaler = StandardScaler()
#scaler.fit(X_train)
#X_train_norm=scaler.transform(X_train)

#Transform test and cross-validation data as well
#X_cv_norm=scaler.transform(X_cv)
#X_test_norm=scaler.transform(X_test)

In [4]:
#Normalize the data so that features are on the same range.
#Tried also standard scaler.
scaler= StandardScaler()
scaler.fit(X_train)
X_train_norm=scaler.transform(X_train)

#Transform test and cross-validation data as well
X_cv_norm=scaler.transform(X_cv)
X_test_norm=scaler.transform(X_test)

In [5]:
#Create an xgboost classifier.
#n_estimators is the number of trees,
#max_depth number of levels allowed per tree,
#use only 50% of features when building
#each tree.
model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=1
)

model.fit(X_train_norm, y_train, 
        eval_set=[(X_train_norm, y_train), (X_cv_norm, y_cv)])

[0]	validation_0-logloss:0.65651	validation_1-logloss:0.66203
[1]	validation_0-logloss:0.64494	validation_1-logloss:0.65032
[2]	validation_0-logloss:0.63995	validation_1-logloss:0.64635
[3]	validation_0-logloss:0.63123	validation_1-logloss:0.63806
[4]	validation_0-logloss:0.62400	validation_1-logloss:0.63265
[5]	validation_0-logloss:0.61596	validation_1-logloss:0.62614
[6]	validation_0-logloss:0.60900	validation_1-logloss:0.61990
[7]	validation_0-logloss:0.60370	validation_1-logloss:0.61575
[8]	validation_0-logloss:0.59898	validation_1-logloss:0.61200
[9]	validation_0-logloss:0.59589	validation_1-logloss:0.61079
[10]	validation_0-logloss:0.59187	validation_1-logloss:0.60793
[11]	validation_0-logloss:0.58844	validation_1-logloss:0.60546
[12]	validation_0-logloss:0.58643	validation_1-logloss:0.60396
[13]	validation_0-logloss:0.58321	validation_1-logloss:0.60136
[14]	validation_0-logloss:0.58046	validation_1-logloss:0.59969
[15]	validation_0-logloss:0.57828	validation_1-logloss:0.59745
[1

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [6]:
#Routine to make predictions for X_input
def model_predict(X_input):
    '''
     X_input    : (ndarray )  Array of features to predict
     returns yhat a numpy array of 0/1 values corresponding to predictions.
    '''    
    #Making predictions.
    prediction = model.predict(X_input)
    
    #Convert to 0/1 predictions
    yhat=np.zeros(len(prediction))
    for i in range(len(prediction)):    
        #Threshold to convert the predictions into TRUE/FALSE outputs.
        if prediction[i] >= 0.5:
            yhat[i] = 1
        else:
            yhat[i] = 0
    return yhat

In [7]:
# Classification Error.
def eval_cat_err(y, yhat):
    """ 
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example           
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
     
        if(y[i]!=yhat[i]):
            incorrect+=1
            
    cerr=incorrect/m
     
    
    return(cerr)

#Error on training set and cross-validation set.
training_cerr = eval_cat_err(y_train, model_predict(X_train_norm))
cv_cerr = eval_cat_err(y_cv, model_predict(X_cv_norm))
print("Training Error")
print(training_cerr)
print("Cross-validation Error")
print(cv_cerr)

Training Error
0.24358100431720064
Cross-validation Error
0.29374337221633084


In [8]:
#Error for reporting model accuracy
accuracy = eval_cat_err(y_test, model_predict(X_test_norm))
print("Testing Accuracy")
print(accuracy)

Testing Accuracy
0.3082627118644068


In [9]:
#Now we test data from 2023
#Training Data.
#Skip the header as it is not a number.
#Each row corresponds to a movie and displays its runtime, IMDB rating, IMDB number of votes,...
X_1 = np.genfromtxt('Hypothesis_X.csv', delimiter=',', skip_header=1)  
m_1, n_1 = X_1.shape
#Each row of y says if 2.5*movie_budget was greater than worldwide_gross. This is the metric we are using to 
#determine if a movie was successful.
y_1 = np.genfromtxt('Hypothesis_Y.csv', delimiter=',', skip_header=1)

#We have to scale before predicting
X_1_test=scaler.transform(X_1)

data2023_cerr = eval_cat_err(y_1, model_predict(X_1_test))


print("Training Error")
print(data2023_cerr)

Training Error
0.363013698630137


In [5]:
!pip install xgboost --user

  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
  Using cached nvidia_nccl_cu12-2.30.4-py3-none-manylinux_2_18_x86_64.whl (300.2 MB)


In [7]:
import site
print(site.getusersitepackages())

/home/sestupin/.local/lib/python3.10/site-packages
